# Evaluate the fine-tuned adapter — Dataset & Column Description

Runs **both** the untuned base model (baseline) **and** the SFT+DPO fine-tuned adapter on the **held-out test datasets** (from `splits.json`) for both tasks, then scores each against the gold descriptions and shows a **side-by-side comparison**:

- **ROUGE-1/2/L** — lexical overlap (CPU, no download)
- **BERTScore** — semantic similarity (downloads an embedding model)
- **Length / verbosity ratio** — generated tokens ÷ reference tokens, flagged above the proposal's brevity threshold

Reuses the **same** task builders and split as `finetune_descriptions.ipynb`, so test inputs match training inputs exactly.

**Both variants run by default** — there is no toggle. The 8B base model is loaded **once**; the adapter is switched on for the fine-tuned pass and switched off (`model.disable_adapter()`) for the baseline pass, so no second model load is needed.

**Where this runs:** generation needs the GPU; the metric math is CPU-light.

## 0. Install (GPU host only)

In [ ]:
%pip install -q --upgrade "transformers>=4.51" "peft>=0.13" "bitsandbytes>=0.45" "datasets>=2.20" "accelerate>=0.34" "rouge-score>=0.1.2" "bert-score>=0.3.13" pandas

## 1. Configuration

In [ ]:
# @dryrun
from pathlib import Path

# ── Evaluation mode ────────────────────────────────────────────────────────
# This notebook evaluates BOTH variants in a single run and compares them:
#   • BASELINE   — the raw untuned Qwen3-8B (adapter disabled)
#   • FINE-TUNED — the SFT+DPO adapter (adapter enabled)
# No toggle needed. The base model is loaded once; the adapter is flipped
# on/off in-place between the two generation passes.

MODEL_ID = "Qwen/Qwen3-8B"

METADATA_PATH = Path("all_metadata.json")
SPLITS_PATH = Path("splits.json")
ADAPTER_PATH = "adapters/qwen3-8b-desc-dpo"

# Per-variant + comparison output paths.
BASELINE_PRED_PATH = Path("baseline_predictions.jsonl")
BASELINE_RESULTS_PATH = Path("baseline_results.json")
FINETUNED_PRED_PATH = Path("eval_predictions.jsonl")
FINETUNED_RESULTS_PATH = Path("eval_results.json")
COMPARISON_PATH = Path("comparison_results.json")

# generation
MAX_NEW_TOKENS = {"dataset_description": 256, "column_description": 64}

# metric toggles
DO_ROUGE, DO_BERTSCORE, DO_LENGTH = True, True, True
BERTSCORE_MODEL = "roberta-large"  # swap for a smaller model if bandwidth is tight
VERBOSITY_THRESHOLD = 1.15  # flag gen/ref length ratio above this

# limit examples while smoke-testing (None = all test examples)
MAX_EVAL_PER_TASK = None

print(f"Model    : {MODEL_ID}")
print(f"Adapter  : {ADAPTER_PATH}")
print(f"Variants : BASELINE (untuned) + FINE-TUNED (SFT+DPO)")

In [ ]:
# --- Colab setup: locate inputs + persist outputs ---
from pathlib import Path

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB  # set False to use Colab's local /content disk instead of Drive
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(
        "/content/drive/MyDrive/MetadataLearn"
    )  # same Drive folder used for training
else:
    PROJECT_DIR = Path(".")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Re-root all paths under PROJECT_DIR.
METADATA_PATH = PROJECT_DIR / "all_metadata.json"
SPLITS_PATH = PROJECT_DIR / "splits.json"
ADAPTER_PATH = str(PROJECT_DIR / "adapters" / "qwen3-8b-desc-dpo")
BASELINE_PRED_PATH = PROJECT_DIR / "baseline_predictions.jsonl"
BASELINE_RESULTS_PATH = PROJECT_DIR / "baseline_results.json"
FINETUNED_PRED_PATH = PROJECT_DIR / "eval_predictions.jsonl"
FINETUNED_RESULTS_PATH = PROJECT_DIR / "eval_results.json"
COMPARISON_PATH = PROJECT_DIR / "comparison_results.json"

# eval needs both the metadata and the split written by the finetune notebook
if (not METADATA_PATH.exists() or not SPLITS_PATH.exists()) and IN_COLAB:
    print("Upload all_metadata.json and/or splits.json:")
    from google.colab import files

    up = files.upload()
    for nm in ("all_metadata.json", "splits.json"):
        if nm in up:
            (PROJECT_DIR / nm).write_bytes(up[nm])

assert (
    METADATA_PATH.exists() and SPLITS_PATH.exists()
), f"Need all_metadata.json + splits.json in {PROJECT_DIR}"

print(f"Adapter  : {ADAPTER_PATH}")
print(f"Variants : BASELINE (untuned) + FINE-TUNED (SFT+DPO)")

## 2. Task definitions (identical to the finetune notebook)

In [ ]:
# @dryrun  — shared task definitions (identical in finetune + evaluate notebooks)
import json, random

SYSTEM_PROMPT = (
    "You are a precise technical writer for open-data catalogs. "
    "Write grounded, concise documentation using only the provided metadata. "
    "Do not speculate or invent statistics. Output only the requested text, "
    "with no preamble, headers, or markdown."
)

MIN_DATASET_DESC_CHARS = (
    40  # skip datasets whose gold description is too thin to learn from
)
MIN_COLUMN_DESC_CHARS = 10  # skip columns with no real description
MAX_SAMPLES_DATASET = (
    4  # sample values shown per column in the dataset-description prompt
)
MAX_SAMPLES_COLUMN = 8  # sample values shown in the column-description prompt
MAX_COLS_IN_PROMPT = 40  # cap columns listed in the dataset-description prompt


def _fmt_samples(samples, n):
    vals = [str(s) for s in (samples or [])[:n] if str(s).strip()]
    return ", ".join(vals) if vals else "(no samples)"


def _column_block(columns):
    lines = []
    for c in columns[:MAX_COLS_IN_PROMPT]:
        lines.append(
            f"- {c.get('name','')} ({c.get('type','')}): e.g. "
            f"{_fmt_samples(c.get('samples'), MAX_SAMPLES_DATASET)}"
        )
    if len(columns) > MAX_COLS_IN_PROMPT:
        lines.append(f"- ... (+{len(columns) - MAX_COLS_IN_PROMPT} more columns)")
    return "\n".join(lines)


def build_dataset_desc_example(rec):
    """One dataset-description training example, or None if the dataset is unusable."""
    desc = (rec.get("description") or "").strip()
    cols = rec.get("columns") or []
    if len(desc) < MIN_DATASET_DESC_CHARS or len(cols) < 2:
        return None
    user = (
        "Task: Write a concise description (2-4 sentences) for the following dataset, "
        "suitable for an open-data catalog. Use only the provided schema and sample values.\n\n"
        f"Dataset name: {rec.get('name','')}\n"
        f"Columns:\n{_column_block(cols)}\n\n"
        "Description:"
    )
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    return {
        "uid": rec.get("uid"),
        "task": "dataset_description",
        "column": None,
        "prompt_messages": prompt_messages,
        "messages": prompt_messages + [{"role": "assistant", "content": desc}],
        "target": desc,
    }


def build_column_desc_examples(rec):
    """All column-description training examples for one dataset."""
    out = []
    name = rec.get("name", "")
    ddesc = (rec.get("description") or "").strip()
    ddesc_short = (
        (ddesc[:300] + "...") if len(ddesc) > 300 else (ddesc or "(none provided)")
    )
    for c in rec.get("columns") or []:
        cdesc = (c.get("description") or "").strip()
        if len(cdesc) < MIN_COLUMN_DESC_CHARS:
            continue
        user = (
            "Task: Write a one-sentence description of the column named below, for the given "
            "dataset. Use only the provided context and sample values.\n\n"
            f"Dataset: {name}\n"
            f"Dataset description: {ddesc_short}\n"
            f"Column name: {c.get('name','')}\n"
            f"Column type: {c.get('type','')}\n"
            f"Sample values: {_fmt_samples(c.get('samples'), MAX_SAMPLES_COLUMN)}\n\n"
            "Column description:"
        )
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        out.append(
            {
                "uid": rec.get("uid"),
                "task": "column_description",
                "column": c.get("name"),
                "prompt_messages": prompt_messages,
                "messages": prompt_messages + [{"role": "assistant", "content": cdesc}],
                "target": cdesc,
            }
        )
    return out


def build_examples_for_uids(records, uids):
    """Return (dataset_desc_examples, column_desc_examples) for the given uids."""
    ds_ex, col_ex = [], []
    for uid in uids:
        rec = records[uid]
        d = build_dataset_desc_example(rec)
        if d:
            ds_ex.append(d)
        col_ex.extend(build_column_desc_examples(rec))
    return ds_ex, col_ex


def split_uids(uids, seed=42, val_frac=0.10, test_frac=0.10):
    """Deterministic held-out-by-dataset split so columns never leak across splits."""
    uids = sorted(uids)
    rng = random.Random(seed)
    rng.shuffle(uids)
    n = len(uids)
    n_test = max(1, round(n * test_frac))
    n_val = max(1, round(n * val_frac))
    return {
        "test": sorted(uids[:n_test]),
        "val": sorted(uids[n_test : n_test + n_val]),
        "train": sorted(uids[n_test + n_val :]),
    }

## 3. Load test split + build eval examples

In [ ]:
# @dryrun
records = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
splits = json.loads(SPLITS_PATH.read_text(encoding="utf-8"))
print({k: len(v) for k, v in splits.items()})

test_ds_ex, test_col_ex = build_examples_for_uids(records, splits["test"])
if MAX_EVAL_PER_TASK:
    test_ds_ex = test_ds_ex[:MAX_EVAL_PER_TASK]
    test_col_ex = test_col_ex[:MAX_EVAL_PER_TASK]
eval_examples = test_ds_ex + test_col_ex
print(
    f"test datasets: {len(splits['test'])} | "
    f"dataset_description: {len(test_ds_ex)} | column_description: {len(test_col_ex)}"
)

## 4. Load base + adapter *(GPU)*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True,  # allow CPU offload for modules that don't fit in GPU
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# Load the adapter once. Both passes share this single model: the adapter is
# ENABLED for the fine-tuned pass and DISABLED for the baseline pass.
adapter_exists = bool(ADAPTER_PATH) and Path(ADAPTER_PATH).exists()
if adapter_exists:
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    has_adapter = hasattr(model, "disable_adapter")
    print("Loaded adapter:", ADAPTER_PATH)
else:
    has_adapter = False
    print(
        f"⚠ No adapter found at {ADAPTER_PATH} — only the BASELINE "
        "will be evaluated (the fine-tuned pass + comparison will be skipped)."
    )
model.eval()

## 5. Generate predictions for both variants *(GPU)*

Runs the test set twice through the **same** loaded model: once with the adapter **disabled** (baseline) and once with it **enabled** (fine-tuned). Both prediction sets are kept in memory (`baseline_predictions`, `finetuned_predictions`) in the same order as `eval_examples`, and each is also written to its own `.jsonl`.

In [ ]:
import json, re, time, contextlib

_THINK_RE = re.compile(
    r"^\s*<think>.*?</think>\s*", re.DOTALL
)  # defensive: drop any leading think block


def generate(prompt_messages, max_new_tokens):
    # enable_thinking=False matches the training-time render (empty think block in the prompt)
    try:
        text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()
    return _THINK_RE.sub("", decoded).strip()


def run_variant(label, adapter_enabled):
    """Generate predictions for every eval example under one model configuration.

    adapter_enabled=False runs inside `model.disable_adapter()` (the baseline);
    adapter_enabled=True runs the model as-loaded (the fine-tuned adapter).
    """
    if adapter_enabled or not has_adapter:
        ctx = contextlib.nullcontext()
    else:
        ctx = model.disable_adapter()  # turn the adapter OFF for the baseline pass

    preds = []
    start_time = time.time()
    print(f"\n=== Generating: {label} ===")
    with ctx:
        for i, ex in enumerate(eval_examples, 1):
            pred = generate(ex["prompt_messages"], MAX_NEW_TOKENS[ex["task"]])
            preds.append(
                {
                    "uid": ex["uid"],
                    "task": ex["task"],
                    "column": ex["column"],
                    "prediction": pred,
                    "reference": ex["target"],
                }
            )
            if i % 10 == 0 or i == len(eval_examples):
                elapsed = time.time() - start_time
                rate = i / elapsed
                remaining = (len(eval_examples) - i) / rate if rate > 0 else 0
                print(
                    f"  [{i:3d}/{len(eval_examples)}] ({rate:.1f} ex/s, ~{remaining:.0f}s remaining)"
                )
    print(f"  done in {time.time() - start_time:.1f}s ({len(preds)} predictions)")
    return preds


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("wrote", path)


# ── Baseline pass (adapter OFF) ──────────────────────────────────────────────
baseline_predictions = run_variant("BASELINE (untuned)", adapter_enabled=False)
write_jsonl(baseline_predictions, BASELINE_PRED_PATH)

# ── Fine-tuned pass (adapter ON) ─────────────────────────────────────────────
if has_adapter:
    finetuned_predictions = run_variant("FINE-TUNED (SFT+DPO)", adapter_enabled=True)
    write_jsonl(finetuned_predictions, FINETUNED_PRED_PATH)
else:
    finetuned_predictions = None
    print("\nNo adapter loaded — skipping the fine-tuned pass.")

## 6. Metrics

Computed overall and broken down by task, **for each variant**. The same `compute_metrics` routine scores the baseline and the fine-tuned predictions independently.

In [ ]:
from collections import defaultdict


def compute_metrics(predictions):
    """Score one variant's predictions: ROUGE, BERTScore, length/verbosity.

    Returns {group: {metric: value}} for group in {overall, dataset_description,
    column_description}.
    """
    by_task = defaultdict(list)
    for p in predictions:
        by_task[p["task"]].append(p)
    groups = {"overall": predictions, **by_task}
    results = {}

    # ---- ROUGE ----
    if DO_ROUGE:
        from rouge_score import rouge_scorer

        scorer = rouge_scorer.RougeScorer(
            ["rouge1", "rouge2", "rougeL"], use_stemmer=True
        )
        for name, rows in groups.items():
            agg = defaultdict(float)
            for p in rows:
                s = scorer.score(p["reference"], p["prediction"])
                for k, v in s.items():
                    agg[k] += v.fmeasure
            n = max(1, len(rows))
            results.setdefault(name, {}).update(
                {k: round(v / n, 4) for k, v in agg.items()}
            )

    # ---- BERTScore ----
    if DO_BERTSCORE:
        import bert_score

        for name, rows in groups.items():
            if name == "overall":
                continue  # scored per task; overall is the example-weighted mean below
            preds = [p["prediction"] for p in rows]
            refs = [p["reference"] for p in rows]
            _, _, f1 = bert_score.score(
                preds,
                refs,
                lang="en",
                model_type=BERTSCORE_MODEL,
                rescale_with_baseline=True,
            )
            results.setdefault(name, {})["bertscore_f1"] = round(float(f1.mean()), 4)
        # example-weighted overall
        tot = sum(results[t].get("bertscore_f1", 0) * len(by_task[t]) for t in by_task)
        results.setdefault("overall", {})["bertscore_f1"] = round(
            tot / max(1, len(predictions)), 4
        )

    # ---- Length / verbosity ratio ----
    if DO_LENGTH:

        def n_tok(s):
            return len(tokenizer.encode(s, add_special_tokens=False))

        for name, rows in groups.items():
            ratios = []
            for p in rows:
                r = n_tok(p["reference"])
                ratios.append(n_tok(p["prediction"]) / r if r else 0.0)
            n = max(1, len(ratios))
            results.setdefault(name, {}).update(
                {
                    "len_ratio_mean": round(sum(ratios) / n, 3),
                    "pct_over_threshold": round(
                        100 * sum(x > VERBOSITY_THRESHOLD for x in ratios) / n, 1
                    ),
                }
            )
    return results


baseline_results = compute_metrics(baseline_predictions)
print("=== BASELINE (untuned) ===")
print(json.dumps(baseline_results, indent=2))

if finetuned_predictions is not None:
    finetuned_results = compute_metrics(finetuned_predictions)
    print("\n=== FINE-TUNED (SFT+DPO) ===")
    print(json.dumps(finetuned_results, indent=2))
else:
    finetuned_results = None

## 7. Per-variant results tables + save

Saves each variant's metrics to its own JSON (`baseline_results.json`, `eval_results.json`) and renders its table.

In [ ]:
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain-python fallback
    display = print


def save_results(results, path, adapter, n_examples):
    path.write_text(
        json.dumps(
            {
                "model_id": MODEL_ID,
                "adapter": adapter,
                "n_examples": n_examples,
                "verbosity_threshold": VERBOSITY_THRESHOLD,
                "metrics": results,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("wrote", path)


# Baseline
save_results(baseline_results, BASELINE_RESULTS_PATH, None, len(baseline_predictions))
print("\nBASELINE (untuned):")
baseline_df = pd.DataFrame(baseline_results).T
baseline_df.index.name = "group"
display(baseline_df)

# Fine-tuned
if finetuned_results is not None:
    save_results(
        finetuned_results,
        FINETUNED_RESULTS_PATH,
        ADAPTER_PATH,
        len(finetuned_predictions),
    )
    print("\nFINE-TUNED (SFT+DPO):")
    finetuned_df = pd.DataFrame(finetuned_results).T
    finetuned_df.index.name = "group"
    display(finetuned_df)

## 8. Side-by-side metric comparison

Baseline vs. fine-tuned with the **delta** (`fine_tuned − baseline`), broken out three ways so you can see where the adapter helps *per task*, not just on the blended average:

- **OVERALL** — dataset + column descriptions combined
- **DATASET_DESCRIPTION** — only the dataset-level task
- **COLUMN_DESCRIPTION** — only the column-level task

Each gets its own table. For ROUGE and BERTScore, higher is better — a positive delta is an improvement. For `len_ratio_mean`, closer to **1.0** is better (≈ matching reference length); for `pct_over_threshold`, lower is better. The full breakdown (all groups, both variants) is saved to `comparison_results.json`.

In [ ]:
import pandas as pd
from collections import Counter

_GROUP_ORDER = ["overall", "dataset_description", "column_description"]
_GROUP_LABELS = {
    "overall": "OVERALL  (dataset + column combined)",
    "dataset_description": "DATASET_DESCRIPTION  (dataset-level task only)",
    "column_description": "COLUMN_DESCRIPTION  (column-level task only)",
}
_METRIC_ORDER = [
    "rouge1",
    "rouge2",
    "rougeL",
    "bertscore_f1",
    "len_ratio_mean",
    "pct_over_threshold",
]


def group_compare(group, baseline_results, finetuned_results):
    """baseline / fine_tuned / delta table for a single group (metric = index)."""
    b = baseline_results.get(group, {})
    f = finetuned_results.get(group, {})
    rows = []
    for m in _METRIC_ORDER:
        if m not in b and m not in f:
            continue
        bv, fv = b.get(m), f.get(m)
        delta = (
            round(fv - bv, 4)
            if isinstance(bv, (int, float)) and isinstance(fv, (int, float))
            else None
        )
        rows.append({"metric": m, "baseline": bv, "fine_tuned": fv, "delta": delta})
    return pd.DataFrame(rows).set_index("metric")


if finetuned_results is not None:
    # Persist the full breakdown (all groups, both variants).
    COMPARISON_PATH.write_text(
        json.dumps(
            {
                "model_id": MODEL_ID,
                "adapter": ADAPTER_PATH,
                "n_examples": len(baseline_predictions),
                "verbosity_threshold": VERBOSITY_THRESHOLD,
                "baseline": baseline_results,
                "fine_tuned": finetuned_results,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("wrote", COMPARISON_PATH)

    # One labelled side-by-side table per group: combined, then each task alone.
    task_counts = Counter(p["task"] for p in baseline_predictions)
    for g in _GROUP_ORDER:
        if g not in baseline_results and g not in finetuned_results:
            continue
        n = len(baseline_predictions) if g == "overall" else task_counts.get(g, 0)
        print(f"\n===== {_GROUP_LABELS[g]}  (n={n}) =====")
        display(group_compare(g, baseline_results, finetuned_results))
else:
    print("No fine-tuned variant available — comparison skipped.")

## 9. Inspect predictions: base vs fine-tuned vs gold

Shows, side by side, the **untuned base model's** output, the **fine-tuned adapter's** output, and the **gold** reference for the *identical* prompt. Both prediction sets were already generated in step 5 (in the same order as `eval_examples`), so this just zips them together — no re-generation needed.

In [ ]:
# Side-by-side: base (untuned) vs fine-tuned vs gold, using the predictions
# already generated in step 5 (same order as eval_examples — no re-generation).

# How many examples to show per task (more column examples since they're short).
N_SHOW = {"dataset_description": 3, "column_description": 10}

if finetuned_predictions is not None:
    for task in ("dataset_description", "column_description"):
        triples = [
            (ex, bp, fp)
            for ex, bp, fp in zip(
                eval_examples, baseline_predictions, finetuned_predictions
            )
            if ex["task"] == task
        ][: N_SHOW[task]]
        print(f"\n########## {task} ({len(triples)} shown) ##########")
        for ex, bp, fp in triples:
            tag = f"{ex['uid']}" + (f"/{ex['column']}" if ex["column"] else "")
            print(f"\n[{tag}]")
            print("BASE      :", bp["prediction"][:300])
            print("FINE-TUNED:", fp["prediction"][:300])
            print("GOLD      :", ex["target"][:300])
else:
    print("No adapter loaded — showing base (untuned) vs gold only.\n")
    for task in ("dataset_description", "column_description"):
        pairs = [
            (ex, bp)
            for ex, bp in zip(eval_examples, baseline_predictions)
            if ex["task"] == task
        ][: N_SHOW[task]]
        print(f"\n########## {task} ({len(pairs)} shown) ##########")
        for ex, bp in pairs:
            tag = f"{ex['uid']}" + (f"/{ex['column']}" if ex["column"] else "")
            print(f"\n[{tag}]")
            print("BASE:", bp["prediction"][:300])
            print("GOLD:", ex["target"][:300])